# Pipeline Xử Lý Tiền Xử Lý Transcript & Trích Xuất Ngữ Nghĩa (LifeMate)

Notebook này xây dựng các **logic tiền xử lý (Preprocessing & Feature Extraction)** cốt lõi cho văn bản giọng nói tiếng Việt (`transcript` từ ASR / `audio_text` từ dataset) trước khi đưa vào mô hình LLM (`Qwen3-4B`).

### Các bước tiền xử lý chuẩn hóa bao gồm:
1. **Chuẩn hóa Văn bản Tiếng Việt Căn bản (Text Normalization)**:
   - Chuẩn hóa Unicode dạng **NFC** (sửa lỗi gõ dấu tổ hợp NFD phổ biến từ ASR).
   - Sửa lỗi khoảng trắng thừa, dấu câu lệch lạc, chuyển chữ thường/hoa có kiểm soát.
2. **Chuẩn hóa Thời gian & Con số Nói Tiếng Việt (Spoken Datetime & Numbers Normalization)**:
   - Quy đổi cách nói giờ tiếng Việt (*"6h rưỡi"*, *"sáu giờ ba mươi"*, *"21h kém 15"*, *"8h tối"*) $\rightarrow$ Chuẩn ISO-8601 (`06:30:00`, `20:45:00`).
   - Quy đổi cách nói ngày tương đối (*"sáng mai"*, *"tối nay"*, *"thứ 6 tuần tới"*) kết hợp với `context.current_time`.
   - Quy đổi tiền tệ / con số nói (*"năm trăm nghìn"*, *"2 triệu"*, *"50k"*) $\rightarrow$ Số nguyên (`500000`, `2000000`, `50000`).
3. **Phát hiện & Bóc tách Tín hiệu Ngữ nghĩa Hội thoại (Dialogue Signals Extractor - 0ms Overhead)**:
   - Phát hiện đại từ chỉ định / câu nói tỉnh lược (*"đơn đó"*, *"lịch vừa nãy"*, *"người đó"*) $\rightarrow$ Cờ `has_anaphora` (DUR).
   - Phát hiện đổi ý định / cập nhật (*"à thôi"*, *"đổi lại"*, *"dời sang"*, *"hủy giúp"*) $\rightarrow$ Cờ `has_supersedes`.
   - Bóc tách liên kết thực thể (Entity Mentions) để chuẩn bị cho chuỗi Read-Before-Write (HEART).

## 1. Import Thư Viện & Khởi Tạo Môi Trường

In [22]:
import re
import json
import unicodedata
from datetime import datetime, timedelta
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

## 2. Bước 1: Chuẩn Hóa Văn Bản Tiếng Việt Căn Bản (Text Normalization)

Mô hình ASR (PhoWhisper) và các bộ gõ tiếng Việt thường sinh ra ký tự Unicode tổ hợp (NFD) hoặc khoảng trắng bất thường. Bước này đưa toàn bộ văn bản về chuẩn NFC duy nhất.

In [23]:
def normalize_vietnamese_text(text: str) -> str:
    """
    Chuẩn hóa căn bản cho transcript tiếng Việt:
    1. Chuyển Unicode sang dạng NFC.
    2. Loại bỏ ký tự điều khiển ẩn và khoảng trắng thừa.
    3. Chuẩn hóa dấu nháy kép, nháy đơn, gạch nối.
    """
    if not text or not isinstance(text, str):
        return ""
    
    # 1. Unicode NFC
    text = unicodedata.normalize("NFC", text)
    
    # 2. Chuẩn hóa dấu ngoặc, dấu nháy
    text = re.sub(r'[""«»]', '"', text)
    text = re.sub(r"[''‚‛]", "'", text)
    text = re.sub(r'[–—―]', '-', text)
    
    # 3. Chuẩn hóa khoảng trắng quanh dấu câu
    text = re.sub(r'\s*([,.?!;:])\s*', r'\1 ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Test thử nghiệm
test_raw = "  Bật   đèn   phòng   khách  ,  rồi   báo   thức  6h   sáng !  "
cleaned = normalize_vietnamese_text(test_raw)
print(f"Raw:     '{test_raw}'")
print(f"Cleaned: '{cleaned}'")

Raw:     '  Bật   đèn   phòng   khách  ,  rồi   báo   thức  6h   sáng !  '
Cleaned: 'Bật đèn phòng khách, rồi báo thức 6h sáng!'


## 3. Bước 2: Chuẩn Hóa Thời Gian Tiếng Việt Nói (Spoken Datetime Normalizer)

Trong tiếng Việt, người dùng nói giờ rất đa dạng:
- *"6h30"*, *"6 giờ rưỡi"*, *"6 rưỡi sáng"*, *"18h30"*
- *"7 giờ kém 15"*, *"9h kém 20 tối"*
- *"sáng mai"*, *"tối nay"*, *"chiều ngày kia"*

Hàm dưới đây nhận diện và quy đổi các mốc này sang định dạng chuẩn `HH:MM:SS` hoặc ISO-8601 kết hợp với `context.current_time`.

In [24]:
_VI_NUM_DICT = {
    'một': 1, 'hai': 2, 'ba': 3, 'bốn': 4, 'năm': 5,
    'sáu': 6, 'bảy': 7, 'tám': 8, 'chín': 9, 'mười': 10,
    'mười một': 11, 'mười hai': 12, 'mười lăm': 15,
    'hai mươi': 20, 'hai mươi lăm': 25, 'ba mươi': 30, 'bốn mươi lăm': 45
}

def normalize_spoken_time(text: str) -> Optional[Tuple[int, int]]:
    """
    Trích xuất (giờ, phút) từ câu nói tiếng Việt:
    Ví dụ: '6 giờ rưỡi sáng' -> (6, 30)
           '8 giờ tối' -> (20, 0)
           '7 giờ kém 15 tối' -> (18, 45)
    """
    text_lower = text.lower()
    is_pm = any(w in text_lower for w in ['chiều', 'tối', 'đêm', 'pm'])
    is_am = any(w in text_lower for w in ['sáng', 'am'])
    
    # 1. Dạng: 6h30, 6:30, 18h
    m1 = re.search(r'\b(\d{1,2})\s*(?:h|:|giờ)\s*(\d{1,2})?\b', text_lower)
    if m1:
        hour = int(m1.group(1))
        minute = int(m1.group(2)) if m1.group(2) else 0
        if 'rưỡi' in text_lower:
            minute = 30
        if is_pm and hour < 12:
            hour += 12
        elif is_am and hour == 12:
            hour = 0
        return hour, minute
        
    # 2. Dạng: 7 giờ kém 15, 8h kém 20
    m2 = re.search(r'\b(\d{1,2})\s*(?:h|giờ)\s*kém\s*(\d{1,2})\b', text_lower)
    if m2:
        hour = int(m2.group(1))
        sub_min = int(m2.group(2))
        # Trừ phút
        minute = (60 - sub_min) % 60
        hour = hour - 1 if sub_min > 0 else hour
        if is_pm and hour < 12:
            hour += 12
        return hour, minute
        
    # 3. Dạng chữ: sáu giờ ba mươi, bảy giờ rưỡi
    m3 = re.search(r'\b(một|hai|ba|bốn|sáu|bảy|tám|chín|mười|mười một|mười hai)\s*giờ(?:\s*(rưỡi|ba mươi|mười lăm|bốn mươi lăm))?\b', text_lower)
    if m3:
        hour = _VI_NUM_DICT.get(m3.group(1), 0)
        min_word = m3.group(2)
        minute = 30 if min_word in ['rưỡi', 'ba mươi'] else (15 if min_word == 'mười lăm' else 0)
        if is_pm and hour < 12:
            hour += 12
        return hour, minute
        
    return None

# Kiểm thử các dạng thời gian nói tiếng Việt
test_times = [
    "Đặt báo thức 6h rưỡi sáng",
    "Nhắc tôi lúc 7h kém 15 tối",
    "Họp vào lúc 14:45 chiều nay",
    "Báo thức tám giờ tối"
]

for t in test_times:
    res = normalize_spoken_time(t)
    if res:
        print(f"'{t}' -> {res[0]:02d}:{res[1]:02d}:00")

'Đặt báo thức 6h rưỡi sáng' -> 06:30:00
'Nhắc tôi lúc 7h kém 15 tối' -> 19:00:00
'Họp vào lúc 14:45 chiều nay' -> 14:45:00
'Báo thức tám giờ tối' -> 20:00:00


## 4. Bước 3: Chuẩn Hóa Ngày Tương Đối Kết Hợp `current_time`

Quy đổi các cụm từ: *"hôm nay"*, *"ngày mai"*, *"sáng mai"*, *"ngày kia"* thành ngày chuẩn `YYYY-MM-DD` dựa trên mốc thời gian của thiết bị.

In [25]:
def resolve_relative_date(text: str, current_time_iso: str) -> str:
    """
    Quy đổi ngày tương đối từ câu nói sang chuẩn YYYY-MM-DD
    dựa trên mốc context.current_time của mẫu.
    """
    try:
        # Parse ISO-8601 (ví dụ: '2026-10-25T08:30:00+07:00')
        base_dt = datetime.fromisoformat(current_time_iso)
    except Exception:
        base_dt = datetime.now()
        
    text_lower = text.lower()
    target_date = base_dt.date()
    
    if any(w in text_lower for w in ['ngày mai', 'mai', 'sáng mai', 'tối mai']):
        target_date += timedelta(days=1)
    elif any(w in text_lower for w in ['ngày kia', 'hôm kia']):
        target_date += timedelta(days=2)
    elif 'hôm nay' in text_lower or 'tối nay' in text_lower or 'chiều nay' in text_lower:
        target_date = base_dt.date()
        
    return target_date.strftime("%Y-%m-%d")

# Test thử nghiệm với mốc context thời gian giả định
sample_context_time = "2026-11-20T21:40:00+07:00"
print("Mốc hiện tại:", sample_context_time)
print("'Đặt báo thức 7h sáng mai' -> Ngày:", resolve_relative_date("Đặt báo thức 7h sáng mai", sample_context_time))
print("'Xem lịch ngày kia có gì'   -> Ngày:", resolve_relative_date("Xem lịch ngày kia có gì", sample_context_time))
print("'Tối nay đi ăn lẩu'          -> Ngày:", resolve_relative_date("Tối nay đi ăn lẩu", sample_context_time))

Mốc hiện tại: 2026-11-20T21:40:00+07:00
'Đặt báo thức 7h sáng mai' -> Ngày: 2026-11-21
'Xem lịch ngày kia có gì'   -> Ngày: 2026-11-22
'Tối nay đi ăn lẩu'          -> Ngày: 2026-11-20


## 5. Bước 4: Chuẩn Hóa Số Lượng & Tiền Tệ Nói (Spoken Currencies & Numbers)

Trong các ứng dụng Ngân hàng (`banking`), Ví điện tử (`ewallet`), Đi chợ (`grocery`), Mua sắm (`ecommerce`):
Người dùng thường nói: *"năm trăm nghìn"*, *"2 triệu"*, *"50k"*, *"1 lít"*, *"2 củ"*.

In [26]:
def extract_currency_amount(text: str) -> Optional[int]:
    """
    Quy đổi tiền tệ tiếng Việt nói sang số nguyên VND:
    Ví dụ: 'chuyển 500k' -> 500000
           'chuyển hai triệu rưỡi' -> 2500000
           'nạp 50 nghìn' -> 50000
    """
    text_lower = text.lower()
    
    # 1. Dạng số kèm k: 500k, 50k, 100k
    m_k = re.search(r'\b(\d+)\s*(?:k|nghìn|ngàn)\b', text_lower)
    if m_k:
        return int(m_k.group(1)) * 1000
        
    # 2. Dạng triệu: 2 triệu, 1.5 triệu, 2 triệu rưỡi
    m_tr = re.search(r'\b(\d+(?:[.,]\d+)?)\s*(?:triệu|tr|củ)\b', text_lower)
    if m_tr:
        val_str = m_tr.group(1).replace(',', '.')
        val = float(val_str) * 1000000
        if 'rưỡi' in text_lower:
            val += 500000
        return int(val)
        
    # 3. Dạng chữ triệu: hai triệu, hai triệu rưỡi, ba triệu, một triệu, hai củ
    _VI_WORDS = {'một': 1, 'hai': 2, 'ba': 3, 'bốn': 4, 'năm': 5, 'sáu': 6, 'bảy': 7, 'tám': 8, 'chín': 9, 'mười': 10}
    m_w_tr = re.search(r'\b(một|hai|ba|bốn|năm|sáu|bảy|tám|chín|mười)\s*(?:triệu|tr|củ)\b', text_lower)
    if m_w_tr:
        val = _VI_WORDS.get(m_w_tr.group(1), 0) * 1000000
        if 'rưỡi' in text_lower:
            val += 500000
        return int(val)
        
    # 4. Nửa triệu / nửa củ
    if re.search(r'\bnửa\s*(?:triệu|củ)\b', text_lower):
        return 500000
        
    # 5. Dạng chữ trăm nghìn: năm trăm nghìn, hai trăm ngàn, một trăm nghìn
    m_tram = re.search(r'\b(một|hai|ba|bốn|năm|sáu|bảy|tám|chín)\s*trăm\s*(?:nghìn|ngàn|k)\b', text_lower)
    if m_tram:
        return _VI_WORDS.get(m_tram.group(1), 0) * 100000
        
    # 6. Dạng chữ chục nghìn: năm mươi nghìn, hai mươi nghìn, ba mươi ngàn
    m_chuc = re.search(r'\b(mười|hai mươi|ba mươi|bốn mươi|năm mươi|sáu mươi|bảy mươi|tám mươi|chín mươi)\s*(?:nghìn|ngàn|k)\b', text_lower)
    if m_chuc:
        _chuc_map = {'mười': 10000, 'hai mươi': 20000, 'ba mươi': 30000, 'bốn mươi': 40000, 'năm mươi': 50000, 'sáu mươi': 60000, 'bảy mươi': 70000, 'tám mươi': 80000, 'chín mươi': 90000}
        return _chuc_map.get(m_chuc.group(1))
        
    return None

# Kiểm thử
test_amounts = [
    "Chuyển 500k cho mẹ",
    "Nạp 50 nghìn tiền điện thoại",
    "Chuyển khoản 2.5 triệu đóng học phí",
    "Bắn hai triệu rưỡi vào tài khoản này"
]
for a in test_amounts:
    amt = extract_currency_amount(a)
    if amt is not None:
        print(f"'{a}' -> Số tiền: {amt:,} VND")
    else:
        print(f"'{a}' -> Không phát hiện số tiền")

'Chuyển 500k cho mẹ' -> Số tiền: 500,000 VND
'Nạp 50 nghìn tiền điện thoại' -> Số tiền: 50,000 VND
'Chuyển khoản 2.5 triệu đóng học phí' -> Số tiền: 2,500,000 VND
'Bắn hai triệu rưỡi vào tài khoản này' -> Số tiền: 2,500,000 VND


## 6. Bước 5: Bộ Trích Xuất Tín Hiệu Đồ Thị Hội Thoại (Dialogue Graph Signals Extractor)

Đây là thành phần cốt lõi được xây dựng trong `promt/prompts.py` (chạy hoàn toàn bằng biểu thức chính quy **0ms overhead**, không gọi thêm LLM phụ):
- Phát hiện đại từ chỉ định / câu nói tỉnh lược (`has_anaphora` $\rightarrow$ Cạnh `COREFERS_TO`).
- Phát hiện yêu cầu đổi ý định (`has_supersedes` $\rightarrow$ Cạnh `SUPERSEDES`).
- Thu thập các mã ID đã xuất hiện trong `history`.

In [27]:
_VI_ANAPHORA_PATTERNS = re.compile(
    r'\b(vừa nãy|hồi nãy|lúc nãy|ở trên|cái đó|đơn đó|lịch đó|báo thức đó|'\
    r'nhắc nhở đó|ghi chú đó|người đó|anh ấy|chị ấy|bạn ấy|họ|tiếp theo|'\
    r'như cũ|như vậy|của tôi vừa tạo|vừa đặt)\b',
    re.IGNORECASE
)

_VI_SUPERSEDES_PATTERNS = re.compile(
    r'\b(đổi lại|dời sang|dời lại|chuyển sang|sửa lại|cập nhật lại|'\
    r'thay đổi|hủy giúp|hủy đơn|xóa lịch|xóa báo thức|thôi không|'\
    r'khoan đã|à nhầm|sửa thành)\b',
    re.IGNORECASE
)

def extract_dialogue_graph_signals(transcript: str, history: Optional[List[Dict[str, Any]]] = None) -> Dict[str, Any]:
    """
    Trích xuất nhanh các cạnh quan hệ ngữ nghĩa hội thoại (0ms overhead):
    - has_anaphora: Có chứa đại từ chỉ định/tỉnh lược không?
    - has_supersedes: Có yêu cầu sửa/dời/hủy ý định trước không?
    - prior_tool_apps: Danh sách các App đã từng gọi trong history.
    - history_entity_ids: Các ID thực thể đã xuất hiện trong history.
    """
    text = (transcript or "").strip()
    anaphora_matches = [m.group(0).lower() for m in _VI_ANAPHORA_PATTERNS.finditer(text)]
    supersedes_matches = [m.group(0).lower() for m in _VI_SUPERSEDES_PATTERNS.finditer(text)]
    
    prior_tool_apps: List[str] = []
    history_entity_ids: Dict[str, List[str]] = {}
    
    for turn in history or []:
        if not isinstance(turn, dict):
            continue
        for tc in turn.get("tool_calls") or []:
            if not isinstance(tc, dict):
                continue
            tname = str(tc.get("name") or "")
            if "." in tname:
                app = tname.split(".", 1)[0]
                if app not in prior_tool_apps:
                    prior_tool_apps.append(app)
            args = tc.get("arguments")
            if isinstance(args, dict):
                for k, v in args.items():
                    if (k == "id" or k.endswith("_id")) and isinstance(v, (str, int)) and str(v).strip():
                        bucket = history_entity_ids.setdefault(k, [])
                        sval = str(v).strip()
                        if sval not in bucket:
                            bucket.append(sval)
                            
    return {
        "cleaned_transcript": normalize_vietnamese_text(text),
        "has_anaphora": bool(anaphora_matches),
        "anaphora_cues": anaphora_matches,
        "has_supersedes": bool(supersedes_matches),
        "supersedes_cues": supersedes_matches,
        "prior_tool_apps": prior_tool_apps,
        "history_entity_ids": history_entity_ids
    }

# Kiểm thử tín hiệu với 1 đoạn hội thoại giả lập
sample_hist = [
    {
        "role": "assistant",
        "tool_calls": [{"name": "alarm.create", "arguments": {"time": "06:00", "alarm_id": "alm_02"}}]
    }
]
query = "À thôi dời cái báo thức vừa nãy sang bảy giờ rưỡi sáng mai giúp tôi"
signals = extract_dialogue_graph_signals(query, sample_hist)
print("Câu nói:", query)
print("Tín hiệu trích xuất được:")
print(json.dumps(signals, ensure_ascii=False, indent=2))

Câu nói: À thôi dời cái báo thức vừa nãy sang bảy giờ rưỡi sáng mai giúp tôi
Tín hiệu trích xuất được:
{
  "cleaned_transcript": "À thôi dời cái báo thức vừa nãy sang bảy giờ rưỡi sáng mai giúp tôi",
  "has_anaphora": true,
  "anaphora_cues": [
    "vừa nãy"
  ],
  "has_supersedes": false,
  "supersedes_cues": [],
  "prior_tool_apps": [
    "alarm"
  ],
  "history_entity_ids": {
    "alarm_id": [
      "alm_02"
    ]
  }
}


## 7. Pipeline Tổng Hợp (End-to-End Transcript Preprocessor)

Hàm tổng hợp duy nhất nhận vào `transcript` thô cùng `context` và `history` của mẫu, trả về một đối tượng dữ liệu sạch sẽ, chuẩn hóa đầy đủ để đưa thẳng vào **Tầng Graph Retrieval** và **LLM Prompt Builder**.

In [28]:
def preprocess_transcript_pipeline(raw_transcript: str, sample_context: Dict[str, Any], history: Optional[List[Dict[str, Any]]] = None) -> Dict[str, Any]:
    """
    Pipeline hoàn chỉnh tiền xử lý Transcript cho LifeMate:
    1. Chuẩn hóa chữ Unicode NFC & dấu câu.
    2. Bóc tách tín hiệu hội thoại (DUR, SUPERSEDES, History IDs).
    3. Nhận diện giờ và quy đổi ngày tương đối.
    4. Nhận diện số tiền nếu có.
    """
    cleaned_text = normalize_vietnamese_text(raw_transcript)
    signals = extract_dialogue_graph_signals(cleaned_text, history)
    
    current_time_iso = sample_context.get('current_time', '')
    resolved_date = resolve_relative_date(cleaned_text, current_time_iso)
    parsed_time = normalize_spoken_time(cleaned_text)
    parsed_amount = extract_currency_amount(cleaned_text)
    
    normalized_iso_dt = None
    if parsed_time:
        normalized_iso_dt = f"{resolved_date}T{parsed_time[0]:02d}:{parsed_time[1]:02d}:00+07:00"
        
    return {
        "raw_input": raw_transcript,
        "normalized_text": cleaned_text,
        "signals": signals,
        "resolved_date": resolved_date,
        "parsed_time": f"{parsed_time[0]:02d}:{parsed_time[1]:02d}:00" if parsed_time else None,
        "normalized_iso_datetime": normalized_iso_dt,
        "parsed_amount_vnd": parsed_amount
    }

# Chạy thử nghiệm Pipeline hoàn chỉnh với mẫu phức tạp
test_query = "À nhầm, chuyển khoản 500k cho anh Nam rồi dời cái lịch vừa nãy sang 8h rưỡi tối mai nhé!"
mock_context = {"current_time": "2026-11-07T10:00:00+07:00"}
mock_history = [
    {"role": "assistant", "tool_calls": [{"name": "calendar.create_event", "arguments": {"event_id": "evt_99"}}]}
]

processed_result = preprocess_transcript_pipeline(test_query, mock_context, mock_history)
print("=== KẾT QUẢ TIỀN XỬ LÝ TOÀN DIỆN ===")
print(json.dumps(processed_result, ensure_ascii=False, indent=2))

=== KẾT QUẢ TIỀN XỬ LÝ TOÀN DIỆN ===
{
  "raw_input": "À nhầm, chuyển khoản 500k cho anh Nam rồi dời cái lịch vừa nãy sang 8h rưỡi tối mai nhé!",
  "normalized_text": "À nhầm, chuyển khoản 500k cho anh Nam rồi dời cái lịch vừa nãy sang 8h rưỡi tối mai nhé!",
  "signals": {
    "cleaned_transcript": "À nhầm, chuyển khoản 500k cho anh Nam rồi dời cái lịch vừa nãy sang 8h rưỡi tối mai nhé!",
    "has_anaphora": true,
    "anaphora_cues": [
      "vừa nãy"
    ],
    "has_supersedes": true,
    "supersedes_cues": [
      "à nhầm"
    ],
    "prior_tool_apps": [
      "calendar"
    ],
    "history_entity_ids": {
      "event_id": [
        "evt_99"
      ]
    }
  },
  "resolved_date": "2026-11-08",
  "parsed_time": "20:30:00",
  "normalized_iso_datetime": "2026-11-08T20:30:00+07:00",
  "parsed_amount_vnd": 500000
}


## 8. Đánh Giá Chất Lượng Trên Các Mẫu Thật Từ `train.jsonl`

Đọc trực tiếp file `train.jsonl` (cả môi trường local và Colab), chọn ra **5 mẫu thực tế đại diện cho nhiều tình huống khác nhau** (hẹn giờ, chuyển tiền, đại từ chỉ định, đổi ý định) để xem kết quả xử lý và đánh giá chất lượng.

In [29]:
# 1. Xác định đường dẫn file train.jsonl (hỗ trợ cả Colab và Local)
possible_train_paths = [
    Path('/content/dataset_lifemate/train+public_test/train/train.jsonl'),
    Path('data/raw/train+public_test/train/train.jsonl'),
    Path('train+public_test/train/train.jsonl'),
    Path('train.jsonl')
]

zip_candidates = [
    Path('/content/drive/MyDrive/ZALO/train+public_test.zip'),
    Path('data/archives/train+public_test.zip'),
    Path('train+public_test.zip')
]
import zipfile
for zpath in zip_candidates:
    if zpath.exists():
        print(f"-> Đang giải nén nhanh train.jsonl từ {zpath}...")
        try:
            with zipfile.ZipFile(zpath, 'r') as zf:
                train_members = [m for m in zf.namelist() if m.endswith('train.jsonl')]
                if train_members:
                    zf.extract(train_members[0], path='.')
                    possible_train_paths.insert(0, Path(train_members[0]))
                    print(f"Đã giải nén xong: {train_members[0]}")
        except Exception as e:
            print('Lỗi giải nén:', e)
        break

train_file = None
for p in possible_train_paths:
    if p.exists():
        train_file = p
        break

if train_file is None:
    print("Chưa tìm thấy file train.jsonl trên ổ đĩa. Đang chạy test trên 5 mẫu giả lập mô phỏng train:")
    real_samples = [
        {
            "id": "train_0010",
            "audio_text": "Đặt giúp tôi một báo thức lúc sáu rưỡi sáng mai nhé.",
            "context": {"current_time": "2026-11-20T21:40:00+07:00"},
            "history": []
        },
        {
            "id": "train_0034",
            "audio_text": "Gửi tin nhắn cho người đó bảo tôi đang trên đường tới rồi.",
            "context": {"current_time": "2026-10-25T08:30:00+07:00"},
            "history": [{"role": "assistant", "tool_calls": [{"name": "contacts.search", "arguments": {"contact_id": "ct_007"}}]}]
        },
        {
            "id": "train_0042",
            "audio_text": "Dời cuộc họp chiều nay với phòng marketing sang ba giờ rưỡi.",
            "context": {"current_time": "2026-11-04T09:00:00+07:00"},
            "history": [{"role": "assistant", "tool_calls": [{"name": "calendar.list_events", "arguments": {"event_id": "evt_88"}}]}]
        },
        {
            "id": "train_0105",
            "audio_text": "Kiểm tra số dư tài khoản xem còn bao nhiêu, sau đó chuyển khoản năm trăm nghìn cho mẹ.",
            "context": {"current_time": "2026-11-15T15:00:00+07:00"},
            "history": []
        },
        {
            "id": "train_0312",
            "audio_text": "Đặt một xe bốn chỗ về nhà, à thôi khoan đã chuyển sang xe bảy chỗ đi cho rộng.",
            "context": {"current_time": "2026-11-12T18:00:00+07:00"},
            "history": []
        }
    ]
else:
    print(f"Đã tìm thấy file: {train_file}")
    real_samples = []
    with open(train_file, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                real_samples.append(json.loads(line))
                if len(real_samples) == 5:
                    break

# 2. Chạy pipeline tiền xử lý và in so sánh chi tiết
print(f"\n{'='*95}")
print(f"KẾT QUẢ ĐÁNH GIÁ TIỀN XỬ LÝ (PREPROCESSING QUALITY CHECK) TRÊN {len(real_samples)} SAMPLES")
print(f"{'='*95}")

for idx, sample in enumerate(real_samples):
    res = preprocess_transcript_pipeline(
        raw_transcript=sample.get('audio_text', ''),
        sample_context=sample.get('context', {}),
        history=sample.get('history', [])
    )
    
    print(f"\n[SAMPLE {idx + 1}] ID: {sample.get('id')}")
    print(f"  • Input gốc (audio_text)     : \"{res['raw_input']}\"")
    print(f"  • Text chuẩn hóa (NFC)       : \"{res['normalized_text']}\"")
    print(f"  • Ngày quy đổi (Date)        : {res['resolved_date']}")
    print(f"  • Giờ trích xuất (Time)      : {res['parsed_time']}")
    print(f"  • Thời gian ISO-8601 chuẩn   : {res['normalized_iso_datetime']}")
    print(f"  • Tiền tệ trích xuất (VND)   : {res['parsed_amount_vnd']:,} đ" if res['parsed_amount_vnd'] else "  • Tiền tệ trích xuất (VND)   : Không có")
    print(f"  • Tín hiệu DUR (Đại từ)      : {res['signals']['has_anaphora']} (Từ khóa: {res['signals']['anaphora_cues']})")
    print(f"  • Tín hiệu Đổi ý (SUPERSEDES): {res['signals']['has_supersedes']} (Từ khóa: {res['signals']['supersedes_cues']})")
    print(f"  • ID từ History tích lũy     : {res['signals']['history_entity_ids']}")
    print(f"  • App lịch sử liên quan      : {res['signals']['prior_tool_apps']}")
    print("-" * 95)

-> Đang giải nén nhanh train.jsonl từ /content/drive/MyDrive/ZALO/train+public_test.zip...
Đã giải nén xong: train+public_test/train/train.jsonl
Đã tìm thấy file: train+public_test/train/train.jsonl

KẾT QUẢ ĐÁNH GIÁ TIỀN XỬ LÝ (PREPROCESSING QUALITY CHECK) TRÊN 5 SAMPLES

[SAMPLE 1] ID: train_0001
  • Input gốc (audio_text)     : "Nếu mai mưa thì dời chạy bộ muộn 1 tiếng, nhắn Sếp Dũng giờ mới; không mưa thì thôi."
  • Text chuẩn hóa (NFC)       : "Nếu mai mưa thì dời chạy bộ muộn 1 tiếng, nhắn Sếp Dũng giờ mới; không mưa thì thôi."
  • Ngày quy đổi (Date)        : 2026-11-29
  • Giờ trích xuất (Time)      : None
  • Thời gian ISO-8601 chuẩn   : None
  • Tiền tệ trích xuất (VND)   : Không có
  • Tín hiệu DUR (Đại từ)      : False (Từ khóa: [])
  • Tín hiệu Đổi ý (SUPERSEDES): False (Từ khóa: [])
  • ID từ History tích lũy     : {}
  • App lịch sử liên quan      : []
-----------------------------------------------------------------------------------------------

[SAMPLE 2] ID: train_00